# Notebook 07 â€” Anomaly Detection

Tasks:
- statistical baseline
- Isolation Forest
- optional LOF
- injected-anomaly recovery
- false-positive inspection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

DATA_DIR = "D:/data viewer/data/raw/solar_power_generation"
OUTPUT_DIR = "D:/data viewer/data/processed"

## Load Data

In [ ]:
def parse_datetime(series):
    for fmt in ('%d-%m-%Y %H:%M', '%Y-%m-%d %H:%M:%S'):
        try:
            return pd.to_datetime(series, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(series, format='mixed')

gen1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Generation_Data.csv")
gen2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Generation_Data.csv")
wx1 = pd.read_csv(f"{DATA_DIR}/Plant_1_Weather_Sensor_Data.csv")
wx2 = pd.read_csv(f"{DATA_DIR}/Plant_2_Weather_Sensor_Data.csv")

for df in [gen1, gen2, wx1, wx2]:
    df['DATE_TIME'] = parse_datetime(df['DATE_TIME'])
    df['hour'] = df['DATE_TIME'].dt.hour

print(f"Plant 1 Generation: {len(gen1)} rows")
print(f"Plant 2 Generation: {len(gen2)} rows")
print(f"Plant 1 Weather: {len(wx1)} rows")
print(f"Plant 2 Weather: {len(wx2)} rows")

## Statistical Baselines

In [ ]:
def statistical_baseline(df, column, method='iqr'):
    if method == 'iqr':
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        return (df[column] < lower) | (df[column] > upper)
    elif method == 'zscore':
        mean = df[column].mean()
        std = df[column].std()
        z_scores = np.abs((df[column] - mean) / std)
        return z_scores > 3
    elif method == 'percentile':
        lower = df[column].quantile(0.01)
        upper = df[column].quantile(0.99)
        return (df[column] < lower) | (df[column] > upper)

def rolling_mad_baseline(df, column, window=96, threshold=3.5):
    rolling_median = df[column].rolling(window=window, center=True, min_periods=1).median()
    rolling_mad = (df[column] - rolling_median).abs().rolling(window=window, center=True, min_periods=1).median()
    modified_z_scores = 0.6745 * (df[column] - rolling_median) / (rolling_mad + 1e-10)
    return np.abs(modified_z_scores) > threshold

print("Plant 1 DC_POWER anomalies:")
print(f"  IQR: {statistical_baseline(gen1, 'DC_POWER', 'iqr').sum()}")
print(f"  Z-score: {statistical_baseline(gen1, 'DC_POWER', 'zscore').sum()}")
print(f"  Percentile: {statistical_baseline(gen1, 'DC_POWER', 'percentile').sum()}")
print(f"  Rolling MAD: {rolling_mad_baseline(gen1, 'DC_POWER').sum()}")

## Isolation Forest and LOF

In [ ]:
# Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
iso_anomalies = iso.fit_predict(gen1[['DC_POWER']].fillna(0)) == -1
print(f"Isolation Forest anomalies: {iso_anomalies.sum()}")

# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_anomalies = lof.fit_predict(gen1[['DC_POWER']].fillna(0)) == -1
print(f"LOF anomalies: {lof_anomalies.sum()}")

## Injected Anomaly Recovery

In [ ]:
# Inject synthetic anomalies
np.random.seed(42)
gen1_injected = gen1.copy()
anomaly_indices = np.random.choice(gen1_injected.index, size=50, replace=False)
gen1_injected.loc[anomaly_indices, 'DC_POWER'] = gen1_injected['DC_POWER'].max() * np.random.uniform(2, 5, size=50)

# Detect with rolling MAD
rolling_anomalies = rolling_mad_baseline(gen1_injected, 'DC_POWER')
detected = rolling_anomalies[anomaly_indices].sum()

print(f"Injected anomalies: {len(anomaly_indices)}")
print(f"Detected by rolling MAD: {detected}")
print(f"Recovery rate: {detected/len(anomaly_indices):.2%}")

## False Positive Inspection

In [ ]:
# Inspect rolling MAD anomalies
anomaly_mask = rolling_mad_baseline(gen1, 'DC_POWER')
anomalies = gen1[anomaly_mask]

print(f"Total rolling MAD anomalies: {len(anomalies)}")
print(f"\nAnomaly statistics:")
print(anomalies['DC_POWER'].describe())

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(gen1['DATE_TIME'], gen1['DC_POWER'], alpha=0.5, label='Normal')
ax.scatter(anomalies['DATE_TIME'], anomalies['DC_POWER'], color='red', label='Anomaly', zorder=5)
ax.set_title('Plant 1 DC Power with Anomalies')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Labeled Evaluation on NAB (Real Messy Data)

NAB provides hand-labeled anomaly windows for real-world series (server metrics, machine sensors, taxi demand). We map windows to point labels and score our detectors with precision/recall/F1 — ground truth instead of synthetic-only injection.

In [ ]:
import json

RAW_DIR = "D:/data viewer/data/raw"
REPORT_DIR = "D:/data viewer/reports"
NAB_DIR = f"{RAW_DIR}/nab"
NAB_SERIES = {
    "machine_temp": "realKnownCause/machine_temperature_system_failure.csv",
    "nyc_taxi": "realKnownCause/nyc_taxi.csv",
    "ec2_cpu": "realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv",
    "grok": "realAWSCloudwatch/grok_asg_anomaly.csv",
}
nab_windows = json.load(open(f"{NAB_DIR}/labels/combined_windows.json"))

def load_nab_series(key):
    df = pd.read_csv(f"{NAB_DIR}/{key}", parse_dates=["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["label"] = 0
    for start, end in nab_windows.get(key, []):
        s, e = pd.to_datetime(start), pd.to_datetime(end)
        df.loc[(df["timestamp"] >= s) & (df["timestamp"] <= e), "label"] = 1
    return df

nab_data = {name: load_nab_series(key) for name, key in NAB_SERIES.items()}
for name, df in nab_data.items():
    print(f"{name}: {len(df)} points, {int(df['label'].sum())} labeled anomalies ({df['label'].mean():.2%})")

In [ ]:
nab_results = []
for name, df in nab_data.items():
    y = df["label"].values
    roll = rolling_mad_baseline(df, "value").astype(int)
    iso = IsolationForest(contamination=max(float(y.mean()), 0.01), random_state=42)
    iso_pred = (iso.fit_predict(df[["value"]].fillna(0)) == -1).astype(int)
    nab_results.append({
        "series": name,
        "labeled_anomalies": int(y.sum()),
        "roll_precision": round(precision_score(y, roll, zero_division=0), 3),
        "roll_recall": round(recall_score(y, roll, zero_division=0), 3),
        "roll_f1": round(f1_score(y, roll, zero_division=0), 3),
        "iso_precision": round(precision_score(y, iso_pred, zero_division=0), 3),
        "iso_recall": round(recall_score(y, iso_pred, zero_division=0), 3),
        "iso_f1": round(f1_score(y, iso_pred, zero_division=0), 3),
    })

nab_results_df = pd.DataFrame(nab_results)
print(nab_results_df.to_string(index=False))
nab_results_df.to_csv(f"{OUTPUT_DIR}/nab_labeled_eval.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/nab_labeled_eval.csv")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, (name, df) in zip(axes.flat, nab_data.items()):
    ax.plot(df["timestamp"], df["value"], lw=0.6, label="value")
    anom = df[df["label"] == 1]
    ax.scatter(anom["timestamp"], anom["value"], color="red", s=8, label="labeled anomaly")
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/nab_labeled_anomalies.png", dpi=150)
plt.show()